# Classifying Open-Ended Survey Responses with Local Models (Ollama)

CatLLM can run entirely on your local machine using [Ollama](https://ollama.com/) — no API keys, no cloud costs, full data privacy. This notebook walks through classification with local models.

**What this notebook covers:**
- Installing and setting up Ollama with CatLLM
- Checking server status and listing available models
- Downloading models automatically from within CatLLM
- Basic single-model classification
- Switching between local models
- Prompt strategies (CoT, step-back, context)
- Robustness features: safety saves, row delay, retry controls
- Multi-model local ensemble
- Mixing local + cloud models in one ensemble
- Local summarization
- Connecting to a remote Ollama server

**What you need:**
- Python 3.9+
- `pip install catllm`
- **Ollama installed on your machine** — download from [ollama.com/download](https://ollama.com/download) (CatLLM cannot install Ollama for you, but it *can* download models once Ollama is installed)
- Ollama server running (`ollama serve`)

**How Ollama works with CatLLM:**

Ollama exposes an OpenAI-compatible HTTP API on `localhost:11434`. CatLLM sends classification prompts to this endpoint. Since local models are smaller and less capable at structured output than cloud models, CatLLM automatically uses a **two-step classification** approach for Ollama:
1. **Step 1:** Ask the model to reason about each category (natural language)
2. **Step 2:** Ask the model to format the result as JSON

This produces more accurate results than asking a local model to reason and output JSON simultaneously.

In [ ]:
import catllm as cat
import pandas as pd

## Setup: Installing and Starting Ollama

CatLLM can download **models** for you, but it cannot install the **Ollama application** itself. You need to install Ollama separately first.

**Install Ollama** (one-time setup):
- **macOS / Linux:** `curl -fsSL https://ollama.com/install.sh | sh`
- **macOS (Homebrew):** `brew install ollama`
- **Windows / manual:** Download from [ollama.com/download](https://ollama.com/download)

**Start the server** (must be running before using CatLLM):
```bash
ollama serve
```

On macOS, Ollama may also run as a menu bar app that starts automatically. Once the server is running, CatLLM can check its status, list models, and download new ones.

In [ ]:
# Check if Ollama is running
if cat.check_ollama_running():
    print("Ollama is running!")
else:
    print("Ollama is not running. Start it with: ollama serve")

In [ ]:
# List all installed models
models = cat.list_ollama_models()
print(f"Installed models ({len(models)}):")
for m in models:
    print(f"  - {m}")

In [ ]:
# Check if a specific model is available
print("llama3.2 available:", cat.check_ollama_model("llama3.2"))
print("qwen2.5:7b available:", cat.check_ollama_model("qwen2.5:7b"))

## Downloading Models

CatLLM can download Ollama models for you. `pull_ollama_model()` checks disk space and RAM requirements, shows the estimated download size, and asks for confirmation before downloading.

You can also skip this step and download models from the command line: `ollama pull llama3.2`

In [ ]:
# Download a model (interactive — will ask for confirmation)
# cat.pull_ollama_model("llama3.2")

# Or auto-confirm the download
# cat.pull_ollama_model("llama3.2", auto_confirm=True)

## Sample Data

Here we create some sample survey responses and define the categories we want to extract.

In [ ]:
# Survey responses to categorize
responses = [
    "because i dont like living here",
    "for a bigger house",
    "to be with my wife",
    "got a new job in another state",
    "rent was too expensive so we had to leave",
]

# Categories to extract from each response
categories = [
    "to start living with or to stay with partner/spouse",
    "relationship change (divorce, breakup, etc)",
    "the person had a job or school or career change, including transferred and retired",
    "the person's partner's job or school or career change, including transferred and retired",
    "financial reasons (rent is too expensive, pay raise, etc)",
    "related specifically to features of the home, such as a bigger or smaller yard",
]

## Basic Classification

For Ollama, you need to set `model_source="ollama"` since auto-detection routes models like `llama3.2` to HuggingFace by default. The API key is required by the tuple format but is ignored — use any placeholder string.

**Note:** Local models are slower than cloud APIs. A 7B model on a MacBook typically processes 1–3 rows per second.

In [ ]:
result = cat.classify(
    input_data=responses,
    categories=categories,
    user_model="llama3.2",
    model_source="ollama",
    api_key="not-needed",
)

result.head()

## Auto-Download Missing Models

Set `auto_download=True` and CatLLM will automatically pull any missing Ollama models before starting classification. Without it, you'll get an interactive prompt asking whether to download.

In [ ]:
result_auto = cat.classify(
    input_data=responses,
    categories=categories,
    user_model="qwen2.5:7b",
    model_source="ollama",
    api_key="not-needed",
    auto_download=True,  # Download if not installed
)

result_auto.head()

## Switching Between Local Models

Just change `user_model` to any model you have installed (or want to auto-download). Smaller models are faster but less accurate.

**Recommended models for classification:**

| Model | Size | Speed | Accuracy | Notes |
|-------|------|-------|----------|-------|
| `llama3.2:1b` | 1.3 GB | Fast | Lower | Good for quick tests |
| `llama3.2` | 2.0 GB | Fast | Good | Best speed/accuracy tradeoff |
| `qwen2.5:7b` | 4.7 GB | Medium | Good | Strong multilingual support |
| `llama3.1:8b` | 4.7 GB | Medium | Better | Good general performance |
| `mistral` | 4.1 GB | Medium | Better | Strong reasoning |
| `gemma2:9b` | 5.4 GB | Slower | Better | Google's local model |
| `deepseek-r1` | 4.7 GB | Slower | Better | Strong at reasoning tasks |

In [ ]:
# Try a different model
result_qwen = cat.classify(
    input_data=responses,
    categories=categories,
    user_model="qwen2.5:7b",
    model_source="ollama",
    api_key="not-needed",
)

result_qwen.head()

## Adding Survey Context

Providing the survey question gives the model important context. You can also enable `context_prompt` to add an expert analyst role to the prompt. These help local models understand the task better.

In [ ]:
result_context = cat.classify(
    input_data=responses,
    categories=categories,
    survey_question="Why did you move?",
    context_prompt=True,
    user_model="llama3.2",
    model_source="ollama",
    api_key="not-needed",
)

result_context.head()

## Prompt Strategies

The same prompt strategies that work with cloud models also work with Ollama:

- **`chain_of_thought=True`**: Step-by-step reasoning. Can help local models think more carefully.
- **`context_prompt=True`**: Expert analyst role. Minimal cost, can improve focus.
- **`step_back_prompt=True`**: Broader reasoning first. Makes an extra LLM call per model.

With local models, prompt strategies have a larger relative impact on accuracy than with cloud models, since smaller models benefit more from explicit reasoning guidance.

In [ ]:
result_prompted = cat.classify(
    input_data=responses,
    categories=categories,
    survey_question="Why did you move?",
    chain_of_thought=True,
    context_prompt=True,
    step_back_prompt=True,
    user_model="llama3.2",
    model_source="ollama",
    api_key="not-needed",
)

result_prompted.head()

## Single-Label Mode

By default, CatLLM uses multi-label classification (multiple categories can be "1" per row). Set `multi_label=False` to force the model to pick exactly one best category. This can be easier for local models since it reduces ambiguity.

In [ ]:
result_single = cat.classify(
    input_data=responses,
    categories=categories,
    user_model="llama3.2",
    model_source="ollama",
    api_key="not-needed",
    multi_label=False,
)

result_single.head()

## Safety Saves and Retry Controls

Local models can fail or produce malformed output more often than cloud models. CatLLM provides several robustness features:

- **`safety=True`**: Save progress to CSV after each row. If the process crashes, you won't lose work.
- **`max_retries`**: Retries per individual API call (default 5).
- **`batch_retries`**: After all rows, retry failed (row, model) pairs.
- **`row_delay`**: Pause between rows. Useful if your machine overheats under sustained inference.
- **`fail_strategy`**: `"partial"` keeps successful results; `"strict"` blanks the row on any failure.

In [ ]:
result_safe = cat.classify(
    input_data=responses,
    categories=categories,
    user_model="llama3.2",
    model_source="ollama",
    api_key="not-needed",
    safety=True,
    filename="local_results.csv",
    save_directory="./output",
    max_retries=3,
    batch_retries=2,
    row_delay=0.5,
)

result_safe.head()

## Multi-Model Local Ensemble

Run multiple local models and combine their results with consensus voting. This improves accuracy — where one model gets confused, another may get it right.

**Important:** When all models are Ollama, CatLLM automatically runs them **sequentially** (one at a time) to avoid overloading your machine. Each model shares the same CPU/GPU, so parallelism would hurt performance.

For Ollama models, set `model_source="ollama"` in each tuple. The API key is ignored.

In [ ]:
result_ensemble = cat.classify(
    input_data=responses,
    categories=categories,
    models=[
        ("llama3.2", "ollama", "not-needed"),
        ("qwen2.5:7b", "ollama", "not-needed"),
    ],
    consensus_threshold="unanimous",
    auto_download=True,
)

result_ensemble.head()

## Mixing Local + Cloud Models

You can combine Ollama models with cloud models in the same ensemble. CatLLM handles the routing automatically — cloud models run in parallel while Ollama models run sequentially.

This is useful for:
- Using a cloud model as a "reference" alongside cheaper local models
- Testing whether local models can match cloud accuracy
- Reducing cloud API costs by supplementing with local inference

In [ ]:
result_mixed = cat.classify(
    input_data=responses,
    categories=categories,
    models=[
        ("gpt-4o-mini", "openai", "YOUR_OPENAI_API_KEY"),
        ("llama3.2", "ollama", "not-needed"),
        ("qwen2.5:7b", "ollama", "not-needed"),
    ],
    consensus_threshold="majority",
    auto_download=True,
)

result_mixed.head()

## Summarization with Local Models

`cat.summarize()` works with Ollama too. The same `model_source="ollama"` applies.

In [ ]:
result_summary = cat.summarize(
    input_data=responses,
    description="Survey responses about why people moved",
    instructions="One sentence summary",
    max_length=30,
    user_model="llama3.2",
    model_source="ollama",
    api_key="not-needed",
)

result_summary.head()

## Multi-Model Summarization Ensemble

Run multiple local models for summarization and synthesize a consensus summary.

In [ ]:
result_summary_ensemble = cat.summarize(
    input_data=responses,
    description="Survey responses about why people moved",
    models=[
        ("llama3.2", "ollama", "not-needed"),
        ("qwen2.5:7b", "ollama", "not-needed"),
    ],
    auto_download=True,
)

result_summary_ensemble.head()

## Remote Ollama Servers

If Ollama is running on a different machine (e.g., a GPU server), use `set_ollama_endpoint()` to point CatLLM at it. This only needs to be called once per session.

In [ ]:
# Point CatLLM at a remote Ollama server
# cat.set_ollama_endpoint(host="192.168.1.100", port=11434)

# Verify it's reachable
# print("Remote server running:", cat.check_ollama_running(host="192.168.1.100"))
# print("Available models:", cat.list_ollama_models(host="192.168.1.100"))

# Then classify as normal — all calls now go to the remote server
# result = cat.classify(
#     input_data=responses,
#     categories=categories,
#     user_model="llama3.1:70b",  # Can run larger models on GPU servers
#     model_source="ollama",
#     api_key="not-needed",
# )

## PDF Classification (Text-Only)

Ollama supports PDF classification but only in **text extraction mode** — it cannot process page images. CatLLM automatically uses text mode when the provider is Ollama.

Requires: `pip install catllm[pdf]`

In [ ]:
# Classify text extracted from PDFs
# result_pdf = cat.classify(
#     input_data="/path/to/pdfs/",
#     categories=["Methodology", "Results", "Discussion"],
#     user_model="llama3.2",
#     model_source="ollama",
#     api_key="not-needed",
# )

## Tips for Getting the Best Results with Local Models

1. **Use verbose categories.** Local models benefit even more than cloud models from detailed category descriptions with examples:
   ```python
   # Instead of:
   "Financial reasons"
   # Use:
   "Financial reasons (rent is too expensive, pay raise, lost job, etc)"
   ```

2. **Provide survey context.** Always set `survey_question` when available — it dramatically helps smaller models.

3. **Enable chain-of-thought.** `chain_of_thought=True` has a bigger positive impact on local models than on cloud models.

4. **Use ensembles.** Running 2–3 different local models with `consensus_threshold="unanimous"` can match the accuracy of a single cloud model.

5. **Use safety saves.** Local models produce more malformed output, so `safety=True` protects against losing progress.

6. **Start small.** Test with a small subset before running thousands of rows. `llama3.2` is a good default — fast, small, and surprisingly capable.

7. **Keep categories under 10.** Local models struggle more with large category sets. Use `categories_per_call` to split them into chunks if needed.

## Ollama-Specific Parameter Reference

| Parameter | Usage for Ollama |
|-----------|------------------|
| `model_source` | Must be `"ollama"` (auto-detection routes llama/qwen to HuggingFace) |
| `api_key` | Required by format but ignored. Use `"not-needed"`. |
| `auto_download` | `True` to auto-pull missing models. Default `False` (interactive prompt). |
| `parallel` | Auto-set to `False` when all models are Ollama (sequential execution). |
| `batch_mode` | **Not supported** for Ollama. Batch API is cloud-only. |
| `mode` (PDF) | Only `"text"` works. Ollama cannot process page images. |
| `creativity` | Works as expected. `0.0` for deterministic, higher for variety. |

**Ollama utility functions:**

| Function | Description |
|----------|-------------|
| `cat.check_ollama_running()` | Check if Ollama server is accessible |
| `cat.list_ollama_models()` | List all installed models |
| `cat.check_ollama_model("llama3.2")` | Check if a specific model is installed |
| `cat.pull_ollama_model("llama3.2")` | Download a model (with confirmation) |
| `cat.set_ollama_endpoint(host, port)` | Point CatLLM at a remote Ollama server |

**Supported models:** Any model available on [ollama.com/library](https://ollama.com/library). Common choices: `llama3.2`, `llama3.1:8b`, `qwen2.5:7b`, `mistral`, `gemma2:9b`, `deepseek-r1`, `phi3`.